IMPORTS + CONFIG

In [1]:
import ast
import json
import os
import time
from collections import Counter, defaultdict
from pathlib import Path

import nbformat
import requests
from secret import GITHUB_TOKEN


In [16]:

# -----------------------------------
# GITHUB CONFIG
# -----------------------------------


HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
}

# Better than generic ipynb search
#QUERYS = 'extension:ipynb train_test_split OR StandardScaler'#'extension:ipynb "import pandas"'
QUERYS = [
    'extension:ipynb train_test_split',
    'extension:ipynb sklearn.preprocessing',
    'extension:ipynb LabelEncoder',
    'extension:ipynb OneHotEncoder',
    'extension:ipynb pandas.read_csv',
    'extension:ipynb RandomForestClassifier',
    'extension:ipynb XGBClassifier',
    'extension:ipynb feature engineering',
        ]
MAX_NOTEBOOKS_PER_QUERY = 250

SAVE_DIR = Path("notebooks")#Path("notebooks_sanity_test")
SAVE_DIR.mkdir(exist_ok=True)

In [3]:
transformations = [
    "bin_equal_frequency_2",
    "bin_equal_frequency_5",
    "bin_equal_frequency_10",
    "bin_equal_width_2",
    "bin_equal_width_5",
    "bin_equal_width_10",
    "norm_min_max",
    "norm_log",
    "zscore_clip_3",
    "zscore_filter_3",
    "winsorize",
    "IQR",
    "isolationForest"
]

EXTRACT GITHUB DATA

In [4]:
# -----------------------------------
# SEARCH GITHUB NOTEBOOKS
# -----------------------------------

def search_notebooks(query, page=1):

    url = "https://api.github.com/search/code"

    params = {
        "q": query,
        "per_page": 100,
        "page": page,
    }

    r = requests.get(
        url,
        headers=HEADERS,
        params=params,
    )
    print(r.status_code)
    print(r.text)
    if r.status_code != 200:

        print("GitHub API ERROR")
        print(r.text)

        return []

    data = r.json()

    return data.get("items", [])

# -----------------------------------
# DOWNLOAD NOTEBOOK
# -----------------------------------

def github_raw_url(html_url):

    raw = html_url.replace(
        "github.com",
        "raw.githubusercontent.com"
    )

    raw = raw.replace("/blob/", "/")

    return raw


def download_notebook(item):

    raw_url = github_raw_url(
        item["html_url"]
    )

    try:

        r = requests.get(raw_url)

        if r.status_code != 200:

            print("FAILED:", raw_url)
            return False

        # verify notebook JSON

        try:

            notebook_json = r.json()

        except Exception:

            print(
                "NOT JSON:",
                raw_url
            )

            return False

        # notebook sanity check

        if "cells" not in notebook_json:

            print(
                "NO CELLS:",
                raw_url
            )

            return False

        repo_name = (
            item["repository"]["full_name"]
            .replace("/", "__")
        )

        filename = item["name"]

        out_path = (
            SAVE_DIR /
            f"{repo_name}__{filename}"
        )

        with open(
            out_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                notebook_json,
                f,
            )

        return True

    except Exception as e:

        print("ERROR:", e)

        return False

CRAWL

In [17]:
# -----------------------------------
# CRAWL NOTEBOOKS
# -----------------------------------

def crawl_notebooks():

    downloaded = 0
    page = 1
    downloaded_notebooks = set()
    for query in QUERYS:
        print("\n*************\nQUERY:", query)
        while downloaded < MAX_NOTEBOOKS_PER_QUERY:

            print(f"\nPAGE {page}")

            items = search_notebooks(query, page)

            if not items:
                print("No more results")
                break

            for item in items:
                repo_name = (
                    item["repository"]["full_name"]
                    .replace("/", "__")
                )
                filename = item["name"]
                notebook_name = f"{repo_name}__{filename}"
                if notebook_name in downloaded_notebooks:
                    print("seen this notebook!")
                    continue

                downloaded_notebooks.add(notebook_name)
                success = download_notebook(item)

                if success:

                    downloaded += 1

                    print(
                        f"Downloaded {downloaded}"
                    )

                if downloaded >= MAX_NOTEBOOKS_PER_QUERY:
                    break

                # avoid rate limits
                time.sleep(0.2)

            page += 1

        downloaded = 0
        page = 1

    print("\nDONE")

In [6]:
# -----------------------------------
# LOAD NOTEBOOK CODE CELLS
# -----------------------------------

def extract_code_cells(notebook_path):

    try:

        nb = nbformat.read(
            notebook_path,
            as_version=4,
        )

        cells = [

            c["source"]

            for c in nb.cells

            if c.cell_type == "code"
        ]

        return cells

    except Exception as e:

        print(f"FAILED: {notebook_path}")
        print(e)

        return []

SEMANTIC RULE ENGINE

In [7]:
# -----------------------------------
# SEMANTIC RULE ENGINE
# -----------------------------------

class SemanticPreprocessingVisitor(ast.NodeVisitor):

    def __init__(self):

        self.transforms = []

        # ---------------------------------
        # symbolic quartile tracking
        # ---------------------------------

        self.last_q1 = None
        self.last_q3 = None

    # -----------------------------------
    # HELPERS
    # -----------------------------------

    def resolve_constant(self, node):

        if isinstance(node, ast.Constant):
            return node.value

        return None

    def get_argument(
        self,
        node,
        kw_name,
        position,
    ):

        # keyword arg

        for kw in node.keywords:

            if kw.arg == kw_name:

                return self.resolve_constant(
                    kw.value
                )

        # positional arg

        if len(node.args) > position:

            return self.resolve_constant(
                node.args[position]
            )

        return None

    # -----------------------------------
    # FUNCTION CALLS
    # -----------------------------------

    def visit_Call(self, node):

        # ---------------------------------
        # attribute calls
        # ---------------------------------

        if isinstance(node.func, ast.Attribute):

            attr = node.func.attr.lower()

            # qcut

            if attr == "qcut":

                q_value = self.get_argument(
                    node=node,
                    kw_name="q",
                    position=1,
                )

                if q_value in [2, 5, 10]:

                    self.transforms.append(
                        f"bin_equal_frequency_{q_value}"
                    )

            # cut

            elif attr == "cut":

                bins_value = self.get_argument(
                    node=node,
                    kw_name="bins",
                    position=1,
                )

                if bins_value in [2, 5, 10]:

                    self.transforms.append(
                        f"bin_equal_width_{bins_value}"
                    )

            # winsorize

            elif attr == "winsorize":

                self.transforms.append(
                    "winsorize"
                )

            # explicit iqr function

            elif attr == "iqr":

                self.transforms.append(
                    "IQR"
                )

            elif attr == "zscore":
                self.transforms.append(
                    "zscore"
                )

        # ---------------------------------
        # direct function calls
        # ---------------------------------

        elif isinstance(node.func, ast.Name):

            func_name = node.func.id.lower()

            # IsolationForest

            if func_name == "isolationforest":

                self.transforms.append(
                    "isolationForest"
                )

            # winsorize

            elif func_name == "winsorize":

                self.transforms.append(
                    "winsorize"
                )

            # IQR

            elif func_name == "iqr":

                self.transforms.append(
                    "IQR"
                )

            # MinMaxScaler

            elif func_name in [
                "minmaxscaler",
                "minmax_scale",
            ]:

                self.transforms.append(
                    "norm_min_max"
                )

            elif func_name == "zscore":
                self.transforms.append(
                    "zscore"
                )

        self.generic_visit(node)

    # -----------------------------------
    # ASSIGNMENTS
    # -----------------------------------

    def visit_Assign(self, node):

        # only simple assignments

        if len(node.targets) != 1:

            self.generic_visit(node)
            return

        target = node.targets[0]

        # ---------------------------------
        # variable assignment
        # ---------------------------------

        if isinstance(target, ast.Name):

            var_name = target.id

            # ---------------------------------
            # RHS is function call
            # ---------------------------------

            if isinstance(node.value, ast.Call):

                value = node.value

                # ---------------------------------
                # attribute call
                # ---------------------------------

                if isinstance(
                    value.func,
                    ast.Attribute
                ):

                    attr = value.func.attr.lower()

                    # -------------------------
                    # quantile(.25/.75)
                    # -------------------------

                    if attr == "quantile":

                        q_value = self.get_argument(
                            node=value,
                            kw_name="q",
                            position=0,
                        )

                        # Q1

                        if q_value == 0.25:

                            self.last_q1 = var_name

                        # Q3

                        elif q_value == 0.75:

                            self.last_q3 = var_name

        # ---------------------------------
        # df["col"] = np.log(...)
        # ---------------------------------

        if isinstance(target, ast.Subscript):

            value = node.value

            if isinstance(value, ast.Call):

                if isinstance(
                    value.func,
                    ast.Attribute
                ):

                    attr = value.func.attr.lower()

                    if attr in [
                        "log",
                        "log1p",
                    ]:

                        self.transforms.append(
                            "norm_log"
                        )

        self.generic_visit(node)

    # -----------------------------------
    # BINARY OPERATIONS
    # -----------------------------------

    def visit_BinOp(self, node):

        # subtraction

        if isinstance(node.op, ast.Sub):

            left = node.left
            right = node.right

            # Q3 - Q1

            if (
                isinstance(left, ast.Name)
                and isinstance(right, ast.Name)
            ):

                left_name = left.id
                right_name = right.id

                if (
                    left_name == self.last_q3
                    and right_name == self.last_q1
                ):

                    self.transforms.append(
                        "IQR"
                    )

        self.generic_visit(node)


In [8]:
# -----------------------------------
# EXTRACT TRANSFORMS FROM CODE
# -----------------------------------

def extract_transforms_from_code(code):

    try:
        tree = ast.parse(code)

        visitor = SemanticPreprocessingVisitor()

        visitor.visit(tree)
        return visitor.transforms

    except Exception:
        return []

In [9]:
# -----------------------------------
# PROCESS SINGLE NOTEBOOK
# -----------------------------------

def process_notebook(notebook_path):

    cells = extract_code_cells(
        notebook_path
    )

    notebook_transforms = []

    for cell in cells:

        transforms = (
            extract_transforms_from_code(
                cell
            )
        )

        notebook_transforms.extend(
            transforms
        )

    return notebook_transforms

In [10]:
# -----------------------------------
# ANALYZE ALL NOTEBOOKS
# -----------------------------------

def analyze_corpus():

    transform_counter = Counter()

    transition_counter = defaultdict(Counter)

    notebook_paths = list(SAVE_DIR.rglob("*.ipynb"))

    print(f"Found {len(notebook_paths)} notebooks")

    for idx, notebook_path in enumerate(notebook_paths):
        if idx % 50 == 0:
            print(f"Processing {idx}")

        transforms = process_notebook(notebook_path)
        if transforms:
            print("\n===================")
            print(notebook_path)
            print(transforms)

        # frequency counts
        transform_counter.update(transforms)

        # transitions

        for a, b in zip(transforms[:-1], transforms[1:]):
            transition_counter[a][b] += 1

    # ---------------------------------
    # transform probabilities
    # ---------------------------------

    total = sum(transform_counter.values())

    transform_probabilities = {
        t: c / total for t, c in (transform_counter.items())
    }

    # ---------------------------------
    # transition probabilities
    # ---------------------------------

    transition_probabilities = {}

    for a, next_ops in (transition_counter.items()):
        total_transitions = sum(next_ops.values())
        transition_probabilities[a] = {
            b: c / total_transitions for b, c in (next_ops.items())
        }

    return (
        transform_probabilities,
        transition_probabilities,
    )

RUN CODE

In [18]:
crawl_notebooks()


*************
QUERY: extension:ipynb train_test_split

PAGE 1
200
{"total_count":23376,"incomplete_results":false,"items":[{"name":"Icecream_Revenue_Prediction.ipynb","path":"Icecream_Revenue_Prediction.ipynb","sha":"4de71fa3178e3b9287087405c75b658e782c8fe2","url":"https://api.github.com/repositories/455534229/contents/Icecream_Revenue_Prediction.ipynb?ref=e474e3f09dc35c9a23fb8cffbcba38ab7979f502","git_url":"https://api.github.com/repositories/455534229/git/blobs/4de71fa3178e3b9287087405c75b658e782c8fe2","html_url":"https://github.com/YBIFoundation/Internship/blob/e474e3f09dc35c9a23fb8cffbcba38ab7979f502/Icecream_Revenue_Prediction.ipynb","repository":{"id":455534229,"node_id":"R_kgDOGybmlQ","name":"Internship","full_name":"YBIFoundation/Internship","private":false,"owner":{"login":"YBIFoundation","id":89171747,"node_id":"MDEyOk9yZ2FuaXphdGlvbjg5MTcxNzQ3","avatar_url":"https://avatars.githubusercontent.com/u/89171747?v=4","gravatar_id":"","url":"https://api.github.com/users/YBIFoundat

NameError: name 'downloaded_notebooks' is not defined

In [12]:
(
    transform_probabilities,
    transition_probabilities,
) = analyze_corpus()

Found 2000 notebooks
Processing 0


C:\Users\danie\PycharmProjects\ATE-shifting\.venv\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)
<unknown>:51: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:52: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:53: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:54: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not


notebooks\AbhishekNatani__Neural_network_project__LEC.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

notebooks\Abstract-Dex__Neural_Nets__classification.ipynb
['norm_min_max']
Processing 50

notebooks\adetbekov__ydl-summer-school__stock_solution.ipynb
['norm_min_max']

notebooks\Adi3220__DSBDA__1)Data Wrangling 1.ipynb
['norm_min_max']

notebooks\Aditya-1663__HealthGo__tf.ipynb
['norm_min_max']
Processing 100

notebooks\Alex9667__Week12__.ipynb
['norm_min_max']

notebooks\alphardaniel020-droid__Data-analytics__titanic.ipynb
['norm_min_max']

notebooks\Altaieb-Mohammed__pytorch-tutorial-YouTube-__Ml1.ipynb
['norm_min_max', 'norm_min_max']

notebooks\alyssa-tsh__CryptoMine__new.ipynb
['norm_log']

notebooks\Aman-Vishwakarma1729__Battery_Health_Insights-and_Prediction_for_Electric_Vehicles__E.ipynb
['norm_min_max']
Processing 150

notebooks\amirhosseinkarimi7__predictive_analysis__q3.ipynb
['zscore', 'zscore

<unknown>:39: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.



notebooks\aryan02420__BITS-F464-Machine-Learning__g11.ipynb
['isolationForest', 'isolationForest']

notebooks\Ashaduzzaman12__Machine_learning__06_Performance_Enhancement_and_Feature_Engineering.ipynb
['norm_min_max', 'norm_log']
Processing 250

notebooks\asumanulusoy__recipe_recommender__vm.ipynb
['norm_min_max']

notebooks\atharvabhoite7__Farming_Assistant_Hack-AI-Thon__disease-detection.ipynb
['norm_min_max']

notebooks\avartan007__deeplearning__4.ipynb
['norm_min_max']

notebooks\bahanivissiley__fraude_detection_machine_learning__sn.ipynb
['IQR']

notebooks\BATspock__deeplearning__NAP.ipynb
['norm_min_max']
Processing 300

notebooks\betr0dalf__TIMO__TIMO_NovikovDV_prac5.ipynb
['norm_min_max']

notebooks\better-data-science__TensorFlow__002_TensorFlow_Regression.ipynb
['zscore', 'norm_min_max']

notebooks\BhavaniPaili__FMML-LAB-1__Regression_Lab_2.ipynb
['norm_min_max']


...

<unknown>:16: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:37: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.



notebooks\VISWA68__TSA__tsa_exp9 (1).ipynb
['norm_min_max']

notebooks\vKenjo__dm-final-proj__attrition_analysis.ipynb
['IQR']

notebooks\Vucibatina__eur_minute_by_minute_predictor__EURPredictor.ipynb
['norm_min_max']
Processing 1900

notebooks\willy0222__ML_100day__Day_031_HW_特徵評估.ipynb
['norm_min_max']

notebooks\Winfry__EnergyPredictionMachineLearning__SMART METER .ipynb
['norm_min_max']


<unknown>:26: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:27: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:28: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:30: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:32: SyntaxWarning: "\g" 


notebooks\Xenaroxy__TIL____[01]Mini_proj_lg_obesity.ipynb
['norm_min_max']

notebooks\yashaur__fraud-detection-project__1 EDA.ipynb
['norm_log', 'norm_log', 'norm_log']

notebooks\Yashrajgk__ds__1_DS.ipynb
['norm_min_max']

notebooks\yaskyj__housing-price-regression__Housing Price Regression.ipynb
['norm_log', 'norm_log', 'norm_min_max']
Processing 1950

notebooks\Yorko__mlcourse.ai__project_telecom_response_prediction_MdScntst.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']

notebooks\YYYYMao__2nd-ML100Days__Day_028_HW.ipynb
['norm_min_max']

notebooks\zakill96__pra__ve3.ipynb
['norm_min_max']

notebooks\zhonghaozhan__REAL-IoT__Anomal_E_cicids2017.ipynb
['isolationForest', 'isolationForest', 'isolationForest', 'isolationForest']

notebooks\zilto__IFT6390-Comp1__comp1_dev.ipynb
['norm_log']


<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:62: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:63: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:62: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:61: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.


In [13]:
# -----------------------------------
# PRINT TRANSFORM PROBABILITIES
# -----------------------------------
print("\n=== TRANSFORM PROBABILITIES ===\n")

for transform, prob in sorted(transform_probabilities.items(), key=lambda x: x[1], reverse=True):
    print(f"{transform:30s} {prob:.4f}")

# -----------------------------------
# PRINT TRANSITION PROBABILITIES
# -----------------------------------
print("\n=== TRANSITION PROBABILITIES ===\n")

for transform_a, transitions in (transition_probabilities.items()):
    print(f"\n{transform_a} ->")

    for transform_b, prob in sorted(transitions.items(), key=lambda x: x[1], reverse=True):
        print(f"    {transform_b:30s} {prob:.4f}")


=== TRANSFORM PROBABILITIES ===

norm_min_max                   0.5779
norm_log                       0.2273
IQR                            0.0909
isolationForest                0.0422
zscore                         0.0260
bin_equal_frequency_5          0.0162
bin_equal_frequency_10         0.0130
bin_equal_width_5              0.0065

=== TRANSITION PROBABILITIES ===


norm_log ->
    norm_log                       0.8500
    norm_min_max                   0.1000
    bin_equal_frequency_10         0.0250
    isolationForest                0.0250

norm_min_max ->
    norm_min_max                   0.8936
    norm_log                       0.0638
    zscore                         0.0213
    IQR                            0.0213

zscore ->
    zscore                         0.5000
    IQR                            0.2500
    norm_min_max                   0.2500

isolationForest ->
    isolationForest                1.0000

bin_equal_frequency_10 ->
    bin_equal_frequency_10         

In [14]:
prob_dict = {}
eps = 1e-10
for transform_op in transformations:
    prob_dict[transform_op] = transform_probabilities.get(transform_op, eps)

prob_dict['zscore_clip_3'] = transform_probabilities.get('zscore', eps)
prob_dict['zscore_filter_3'] = transform_probabilities.get('zscore', eps)
print(prob_dict)


{'bin_equal_frequency_2': 1e-10, 'bin_equal_frequency_5': 0.016233766233766232, 'bin_equal_frequency_10': 0.012987012987012988, 'bin_equal_width_2': 1e-10, 'bin_equal_width_5': 0.006493506493506494, 'bin_equal_width_10': 1e-10, 'norm_min_max': 0.577922077922078, 'norm_log': 0.22727272727272727, 'zscore_clip_3': 0.025974025974025976, 'zscore_filter_3': 0.025974025974025976, 'winsorize': 1e-10, 'IQR': 0.09090909090909091, 'isolationForest': 0.04220779220779221}
